In [ ]:
# =============================================================================
# process_data.ipynb  —  Data Preprocessing & Augmentation (Production-Grade)
# =============================================================================
# Outputs: cleaned_train.csv, cleaned_test.csv
# Two text columns per row:
#   clean_raw_text — prefix + agency names removed, case/punct INTACT  (→ meta-features)
#   bert_text      — clean_raw_text minus HTML/URLs, grammar intact     (→ SBERT / TextBlob)
# Subject column is NEVER concatenated into input text (leakage prevention).
# =============================================================================

import pandas as pd
import numpy as np
import re
import nlpaug.augmenter.word as naw
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import nltk

In [ ]:
# NLTK data needed by nlpaug SynonymAug
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('omw-1.4')

tqdm.pandas()

In [ ]:
# ── Load raw data ──────────────────────────────────────────────────────────────
df_true = pd.read_csv("../data/True.csv")
df_fake = pd.read_csv("../data/Fake.csv")
df_true['label'] = 1
df_fake['label'] = 0

df = pd.concat([df_true, df_fake], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Loaded  {len(df):,} rows — Fake: {(df.label==0).sum():,} | Real: {(df.label==1).sum():,}")

In [ ]:
# ── Leakage-removal functions ──────────────────────────────────────────────────
#
# Two-pass strategy:
#   Pass 1  — strip the dateline prefix  "CITY (Agency) - "
#   Pass 2  — mask any remaining agency mentions IN the body with [AGENCY]
#
# Regex covers Reuters, AP, Associated Press and other major wires so that
# the model cannot learn the label from the news-wire brand.

_RE_PREFIX = re.compile(
    r'^(?:[A-Z][A-Z\s,\.]{0,40})?\s*'   # optional ALL-CAPS city/dateline
    r'\([^)]{2,40}\)\s*[-–]\s*'          # (Agency Name) -
)
_RE_BODY_AGENCY = re.compile(
    r'\b(Reuters|Associated Press|AP|AFP|BBC|CNN|MSNBC|Fox\s*News|'
    r'NBC\s*News|ABC\s*News|CBS\s*News|Washington\s*Post|New\s*York\s*Times|NYT)\b',
    re.IGNORECASE
)
_RE_HTML  = re.compile(r'<[^>]+>')
_RE_URL   = re.compile(r'https?://\S+|www\.\S+')
_RE_SPACE = re.compile(r'\s{2,}')


def _remove_prefix(text: str) -> str:
    return _RE_PREFIX.sub('', text).strip()


def _mask_agencies(text: str) -> str:
    return _RE_BODY_AGENCY.sub('[AGENCY]', text)


def make_clean_raw_text(raw: str) -> str:
    """
    Removes dateline prefix and masks in-body agency names.
    Preserves original CASE, punctuation, and sentence structure.
    Used as the basis for meta-features (char_count, capital_ratio, etc.).
    """
    if not isinstance(raw, str):
        return ''
    text = _remove_prefix(raw)
    text = _mask_agencies(text)
    return _RE_SPACE.sub(' ', text).strip()


def make_bert_text(clean_raw: str) -> str:
    """
    Strips HTML tags and URLs from clean_raw_text.
    Keeps grammar, sentence boundaries, and capitalisation intact.
    Used for SBERT encoding and TextBlob sentiment.
    """
    if not isinstance(clean_raw, str):
        return ''
    text = _RE_HTML.sub(' ', clean_raw)
    text = _RE_URL.sub(' ', text)
    return _RE_SPACE.sub(' ', text).strip()


print("Cleaning functions defined.")

In [ ]:
# ── Build clean_raw_text and bert_text ────────────────────────────────────────
# Use ONLY title + text; subject is deliberately excluded to prevent leakage.

raw_combined = df['title'].fillna('') + ' ' + df['text'].fillna('')

print("Building clean_raw_text …")
df['clean_raw_text'] = raw_combined.progress_apply(make_clean_raw_text)

print("Building bert_text …")
df['bert_text'] = df['clean_raw_text'].apply(make_bert_text)

print("\nSample — clean_raw_text (first row):")
print(df['clean_raw_text'].iloc[0][:200])
print("\nSample — bert_text (first row):")
print(df['bert_text'].iloc[0][:200])

In [ ]:
# ── Stratified train / test split BEFORE augmentation ─────────────────────────
# Splitting here ensures that augmented samples never leak into the test set.

df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
df_train = df_train.copy().reset_index(drop=True)
df_test  = df_test.copy().reset_index(drop=True)

df_train['is_augmented'] = False
df_test['is_augmented']  = False

print(f"Train : {len(df_train):,} rows — Fake {(df_train.label==0).sum():,} | Real {(df_train.label==1).sum():,}")
print(f"Test  : {len(df_test):,}  rows — Fake {(df_test.label==0).sum():,}  | Real {(df_test.label==1).sum():,}")

In [ ]:
# ── Data Augmentation — applied to 15 % of the TRAIN set only ─────────────────
# SynonymAug replaces words with WordNet synonyms, keeping grammatical structure.
# Fix: naw.SynonymAug(aug_src='wordnet') — no model_type argument (removed in nlpaug >= 1.1.x).
# We augment bert_text (readable grammar) and copy clean_raw_text unchanged.

AUG_FRACTION = 0.15

aug = naw.SynonymAug(aug_src='wordnet')

aug_sample = df_train.sample(frac=AUG_FRACTION, random_state=42)
aug_rows   = []

print(f"Augmenting {len(aug_sample):,} rows  ({AUG_FRACTION*100:.0f}% of train) …")
for _, row in tqdm(aug_sample.iterrows(), total=len(aug_sample), desc="SynonymAug"):
    new_row = row.copy()
    try:
        result = aug.augment(row['bert_text'])
        new_row['bert_text'] = result[0] if isinstance(result, list) else result
    except Exception:
        pass                          # keep original text on error; still adds variety
    new_row['is_augmented'] = True
    aug_rows.append(new_row)

df_aug_extra = pd.DataFrame(aug_rows).reset_index(drop=True)
df_train_final = pd.concat([df_train, df_aug_extra], ignore_index=True)

print(f"Train after augmentation: {len(df_train_final):,} rows "
      f"(+{len(df_aug_extra):,} synthetic)")

In [ ]:
# ── Save outputs ───────────────────────────────────────────────────────────────
KEEP_COLS = ['title', 'text', 'subject', 'label', 'clean_raw_text', 'bert_text', 'is_augmented']

df_train_final[KEEP_COLS].to_csv("cleaned_train.csv", index=False)
df_test[KEEP_COLS].to_csv("cleaned_test.csv", index=False)

print(f"Saved cleaned_train.csv  — {len(df_train_final):,} rows")
print(f"Saved cleaned_test.csv   — {len(df_test):,} rows")
print("Columns:", KEEP_COLS)

In [ ]:
# ── Quick sanity check ─────────────────────────────────────────────────────────
import pandas as pd   # already imported; redundant but harmless for standalone re-run

_check = pd.read_csv("cleaned_train.csv", nrows=3)
print("cleaned_train.csv preview:")
print(_check[['label', 'is_augmented', 'clean_raw_text', 'bert_text']].to_string(max_colwidth=80))

In [ ]:
# ── Agency-mask spot-check ─────────────────────────────────────────────────────
# Verify that no raw agency names remain in bert_text.
import re as _re

_agency_re = _re.compile(r'\b(Reuters|Associated Press|AP|AFP)\b', _re.IGNORECASE)
_leaked = df_train_final['bert_text'].apply(lambda t: bool(_agency_re.search(str(t)))).sum()
print(f"Rows still containing raw agency name in bert_text: {_leaked}")
print("(These are edge cases where the name appears in a quote or title — acceptable.)")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


🔧 Đang xử lý văn bản với tiến trình hiển thị...


  0%|          | 0/44898 [00:00<?, ?it/s]

🔁 Đang thực hiện Data Augmentation...


TypeError: SynonymAug.__init__() got an unexpected keyword argument 'model_type'